In [1]:
import random
import pandas as pd
from collections import defaultdict
import itertools

In [ ]:
# This function evaluates EV of each action independently
def blackjack_basic_strategy_MC(
        number_of_decks = 6,
        sims = 50000,
        deck_pen = 0.75,
        dealer_hit_soft_17 = True,
        double_allowed = True,
        split_allowed = False, #not included
        surrender_allowed = False, #not included
        insurance_allowed = False, #not included
        blackjack_payout = 1.5
):
    # Card Setup -------------------------------------------------
        bj_rank = ['A','2','3','4','5','6','7','8','9','10','10','10','10']
        single_deck = bj_rank * 4
        game_decks = single_deck * number_of_decks

        # All possible staring hands 
        bj_rank_unique = ['A','2','3','4','5','6','7','8','9','10']
        player_cards = []
        for i in bj_rank_unique:
                for j in bj_rank_unique:
                        if j < i:
                                continue
                        else:
                                player_cards.append([i,j])

        dealer_face = bj_rank_unique.copy()

    # Result Storage -------------------------------------------------
        allowed_move = ['stand', 'hit']

        if double_allowed == True:
                allowed_move.append('double')
        if split_allowed == True:
                allowed_move.append('split')
        if surrender_allowed == True:
                allowed_move.append('surrender')
        if insurance_allowed == True:
                allowed_move.append('insurance')

        results = defaultdict(lambda:{'winning': 0, 'stake': 0, 'count':0})

        def record(dealer_up, player_initial, move, profit, stake):
                key = (dealer_up, ','.join(sorted(player_initial)), move)
                results[key]['winning'] += profit
                results[key]['stake'] += stake
                results[key]['count'] += 1

    # Hand Value -------------------------------------------------
        def hand_values(card_list):
                output = 0
                aces = 0
                for i in card_list:
                        if i == 'A':
                                aces += 1
                                output += 11
                        else:
                                output += int(i)
                while output > 21 and aces > 0:
                        output -= 10
                        aces -= 1
                return output    
        
        def is_soft(hand):
                return 'A' in hand and hand_values(hand) <= 21 and hand_values(hand) - 10 >= 12

    # Dealer Move -------------------------------------------------
        def dealer_move(dealer_cards, deck_slice):
                cards = dealer_cards.copy()
                i = 0
                while True:
                        value = hand_values(cards)
                        if value > 21:
                                return cards, True, i # dealer bust
                        if value >= 17 and (value > 17 or not (dealer_hit_soft_17 and is_soft(cards))):  
                                return cards, False, i
                        if i >= len(deck_slice):
                                return cards, value > 21, i
                        cards.append(deck_slice[i])
                        i += 1       

    # Player Moves -------------------------------------------------
        def stand(player_cards, dealer_cards, deck_slice):
                dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice)
                player_value = hand_values(player_cards)
                dealer_value = hand_values(dealer_fin)
                if dealer_busted or player_value > dealer_value:
                        record(dealer_cards[0], player_cards[:2], 'stand', 1, 1)
                elif player_value == dealer_value: #push
                        record(dealer_cards[0], player_cards[:2], 'stand', 0, 1)
                else:
                        record(dealer_cards[0], player_cards[:2], 'stand', -1, 1)
                
        def hit(player_cards, dealer_cards, deck_slice):
                cards = player_cards.copy()
                cards.append(deck_slice[0])
                player_value = hand_values(cards)
                if player_value > 21:
                        record(dealer_cards[0], player_cards[:2], 'hit', -1, 1)
                else:
                        dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice[1:]) 
                        dealer_value = hand_values(dealer_fin)
                        if dealer_busted or player_value > dealer_value:
                                record(dealer_cards[0], player_cards[:2], 'hit', 1, 1)
                        elif player_value == dealer_value: #push
                                record(dealer_cards[0], player_cards[:2], 'hit', 0, 1)
                        else:
                                record(dealer_cards[0], player_cards[:2], 'hit', -1, 1)                       
       
        def double(player_cards, dealer_cards, deck_slice):
                cards = player_cards.copy()
                cards.append(deck_slice[0])
                player_value = hand_values(cards)
                if player_value > 21:
                        record(dealer_cards[0], player_cards[:2], 'double', -2, 2)
                else:
                        dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice[1:]) 
                        dealer_value = hand_values(dealer_fin)
                        if dealer_busted or player_value > dealer_value:
                                record(dealer_cards[0], player_cards[:2], 'double', 2, 2)
                        elif player_value == dealer_value: #push
                                record(dealer_cards[0], player_cards[:2], 'double', 0, 2)
                        else:
                                record(dealer_cards[0], player_cards[:2], 'double', -2, 2)                
        
        def split(player_cards, dealer_cards, deck_slice):
                return
        def surrender(player_cards, dealer_cards, deck_slice):
                return
        def insurance(player_cards, dealer_cards, deck_slice):
                return

    # Sim -------------------------------------------------

        total_cards = len(game_decks)
        pen_limit = int(total_cards * deck_pen)
        games = 0

        while games < sims:
                current_deck = game_decks.copy()
                random.shuffle(current_deck)
                pos = 0

                while pos < pen_limit:
                        player_initial = [current_deck[pos], current_deck[pos+1]]
                        dealer_cards = [current_deck[pos+2], current_deck[pos+3]]
                        pos += 4
                        deck_slice = current_deck[pos:]

                        #check for blackjack first 
                        if dealer_cards[0] == 'A' and hand_values(dealer_cards) == 21:
                                if hand_values(player_initial) == 21:
                                        record(dealer_cards[0], player_initial[:2], 'blackjack', 0, 1) # push
                                else:
                                        record(dealer_cards[0], player_initial[:2], 'dealer-blackjack', -1, 1) 
                        if hand_values(player_initial) == 21:
                                        if hand_values(dealer_cards) == 21:
                                                record(dealer_cards[0], player_initial[:2], 'blackjack', 0, 1) # push
                                        else:
                                                record(dealer_cards[0], player_initial[:2], 'blackjack', 1 * blackjack_payout, 1) 
                        else:
                                # need to move the deck so the dealer isn't taking the same cards
                                stand(player_initial, dealer_cards, deck_slice)
                                dealer_fin, dealer_busted, num_dealer_cards = dealer_move(dealer_cards, deck_slice[1:]) # untidy, this does not need to be in every function - remove if have time
                                hit(player_initial, dealer_cards, deck_slice)
                                if 'double' in allowed_move:
                                        double(player_initial, dealer_cards, deck_slice)
                                if 'split' in allowed_move:
                                        split(player_initial, dealer_cards, deck_slice)
                                if 'surrender' in allowed_move:
                                        surrender(player_initial, dealer_cards, deck_slice)
                                if 'insurance' in allowed_move:
                                        insurance(player_initial, dealer_cards, deck_slice)                      
                        pos += 1 + num_dealer_cards # for player move and dealer cards         

                games += 1

    # Results Table -------------------------------------------------
        
        results_df = pd.DataFrame(results).T.reset_index()
        results_df.columns = ['dealer_up', 'player_hand', 'move', 'winning', 'stake', 'count']
        results_df['EV'] = results_df['winning'] / results_df['stake']

        ev_table = results_df.pivot_table(
        index=['dealer_up', 'player_hand'],
        columns='move',
        values='EV'
        )

        ev_table['avg_EV'] = ev_table.mean(axis=1)
        ev_table = ev_table.sort_values(by='avg_EV', ascending=True)
        ev_table = ev_table.drop(columns='avg_EV')

        ev_table = ev_table.round(4)
        ev_table = ev_table.sort_index()
        

        return ev_table 



In [3]:
test = blackjack_basic_strategy_MC(
        number_of_decks = 1,
        sims = 50000,
        deck_pen = 0.75,
        dealer_hit_soft_17 = True,
        double_allowed = True,
        split_allowed = False, #not included
        surrender_allowed = False, #not included
        insurance_allowed = False, #not included
        blackjack_payout = 1.5
)

In [42]:
test

move                   blackjack  double     hit   stand
dealer_up player_hand                                   
10        10,10              NaN -0.8429 -0.8429  0.4426
          10,2               NaN -0.4288 -0.4288 -0.5744
          10,3               NaN -0.4158 -0.4158 -0.5889
          10,4               NaN -0.4920 -0.4920 -0.5766
          10,5               NaN -0.5219 -0.5219 -0.5677
...                          ...     ...     ...     ...
A         8,9                NaN -0.7134 -0.7134 -0.6710
          8,A                NaN -0.3876 -0.3876 -0.1628
          9,9                NaN -0.7778 -0.7778 -0.3333
          9,A                NaN -0.3551 -0.3551  0.0471
          A,A                NaN -0.4386 -0.4386 -0.6491

[550 rows x 4 columns]